# Hafta 9-10: Tüm fan ID'leri, 8 kanal ve makine ID sınıflandırıcısı

Bu defter Hafta 1-8 defterinin devamıdır. Yenilikler:

1. **Veri:** fan id_00, 02, 04, 06 × +6 / 0 / −6 dB, **8 kanalın tamamı** (öncekinde yalnızca id_00 ve ilk kanal).
2. **8 kanal birleştirme:** `ch1` (ilk kanal), `ort8` (8 kanal dalga ortalaması), `gort8` (8 kanal güç spektrogramı ortalaması).
3. **Makine ID sınıflandırıcısı** (proje planındaki, yapılmamış model): yalnızca normal seslerle, sesin hangi fana ait olduğunu öğrenen CNN + SpecAugment. Anomali skoru = −log p(doğru ID).
4. **ID-XAI (önerilen, etiketsiz):** XAI öznitelik seçimi arıza etiketi yerine makine kimliği ile yapılır (RF bire karşı diğerleri, yalnız normal kayıt) + kNN, k ∈ {5,10,20,33} topluluğu, 1. kanal + 8 kanal ortalaması öznitelikleri.
5. **Füzyon:** ID sınıflandırıcısı ve öznitelik kNN skorlarının sıra ortalaması (etiket kullanmaz).
6. 3 tohum, tohum **bölünmeyi de** değiştirir; dengeli protokol + tez protokolü; AUC ve pAUC.

Tüm ara dosyalar Drive'da `mimii/hafta9` altında tutulur; oturum koparsa hücreleri tekrar çalıştırmak kaldığı yerden devam eder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q librosa soundfile scikit-learn
import os; os.makedirs('/content/code', exist_ok=True)
WORK = '/content/drive/MyDrive/mimii/hafta9'; os.makedirs(WORK, exist_ok=True); os.chdir(WORK); print(os.getcwd())

## Kod modülleri
`hand.py`: Hafta 7'deki 145 öznitelik. `feats.py`: 8 kanallı kayıttan üç modda log-mel ve öznitelik. `prep_colab.py`: zip'i bütün halde indirip açmadan işler. `exp.py`: modeller. `run.py`: deney koşucusu. `analyze.py`: tablolar ve füzyon. `plots.py`: şekiller.

In [ ]:
%%writefile /content/code/hand.py
import numpy as np, librosa
from scipy import stats as sst, signal as ssig
import multiprocessing as mp

STAT_NAMES = ["mean", "var", "std", "skew", "kurt", "max", "mad", "acf1"]

def seq_stats(x):
    x = np.asarray(x, dtype=np.float64).ravel()
    x = x[np.isfinite(x)]
    sd = x.std()
    ok = sd > 1e-12 and x.size > 3
    return [x.mean(), x.var(), sd,
            sst.skew(x) if ok else 0.0,
            sst.kurtosis(x) if ok else 0.0,
            x.max(), sst.median_abs_deviation(x),
            np.corrcoef(x[:-1], x[1:])[0, 1] if ok else 0.0]

def extract_features_y(y, sr=16000):
    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=512))
    P = S ** 2
    f = {}
    def add(name, seq):
        for sn, v in zip(STAT_NAMES, seq_stats(seq)):
            f[f"{name}_{sn}"] = v
    add("centroid", librosa.feature.spectral_centroid(S=S, sr=sr)[0])
    add("bandwidth", librosa.feature.spectral_bandwidth(S=S, sr=sr, p=1)[0])
    add("spread", librosa.feature.spectral_bandwidth(S=S, sr=sr, p=2)[0])
    add("rolloff", librosa.feature.spectral_rolloff(S=S, sr=sr)[0])
    add("flatness", librosa.feature.spectral_flatness(S=S)[0])
    for i, v in enumerate(librosa.feature.spectral_contrast(S=S, sr=sr, fmin=100, n_bands=6).mean(axis=1)):
        f[f"contrast{i}_mean"] = v
    _, psd = ssig.welch(y, fs=sr, nperseg=1024)
    add("psd", np.log10(psd + 1e-12))
    frames = librosa.util.frame(y, frame_length=1024, hop_length=512)
    add("envelope", np.abs(frames).max(axis=0))
    rms = librosa.feature.rms(S=S, frame_length=1024)[0]
    add("rms", rms)
    add("db", librosa.amplitude_to_db(rms + 1e-10))
    chroma = librosa.feature.chroma_stft(S=P, sr=sr, tuning=0.0)
    for i in range(12):
        f[f"chroma{i}_mean"] = chroma[i].mean()
        f[f"chroma{i}_max"] = chroma[i].max()
    C = np.abs(librosa.cqt(y, sr=sr, hop_length=512, n_bins=48, bins_per_octave=12, fmin=librosa.note_to_hz("C2")))
    add("cqt", librosa.amplitude_to_db(C.mean(axis=0) + 1e-10))
    add("hilbert", np.abs(ssig.hilbert(y))[::16])
    add("zcr", librosa.feature.zero_crossing_rate(y, frame_length=1024, hop_length=512)[0])
    low = librosa.fft_frequencies(sr=sr, n_fft=1024) < 1000
    add("bandratio", P[low].sum(axis=0) / (P.sum(axis=0) + 1e-12))
    add("magnitude", S.mean(axis=0))
    f["sig_skew"] = sst.skew(y)
    f["sig_kurt"] = sst.kurtosis(y)
    return {k: float(np.nan_to_num(v)) for k, v in f.items()}

In [ ]:
%%writefile /content/code/feats.py
"""8 kanallı MIMII kayıtlarından üç modda log-mel çıkarır: ch1 (ilk kanal), ort8 (8 kanal dalga ortalaması), gort8 (8 kanal güç spektrogramı ortalaması)."""
import sys, numpy as np, soundfile as sf, librosa
sys.path.insert(0, str(__import__("pathlib").Path(__file__).parent)); import hand
from pathlib import Path
from multiprocessing import Pool
SR, NFFT, HOP, NMELS = 16000, 1024, 512, 128
MEL = librosa.filters.mel(sr=SR, n_fft=NFFT, n_mels=NMELS)
def pw(y): return np.abs(librosa.stft(y, n_fft=NFFT, hop_length=HOP)) ** 2
def db(P): return librosa.power_to_db(MEL @ P, ref=1.0).astype(np.float16)
def one(p):
    x, _ = sf.read(p, dtype="float32")
    P = np.stack([pw(x[:, c]) for c in range(x.shape[1])])
    m = x.mean(1); h1 = hand.extract_features_y(x[:, 0]); h8 = hand.extract_features_y(m)
    return db(P[0]), db(pw(m)), db(P.mean(0)), list(h1.values()), list(h8.values()), list(h1.keys())
if __name__ == "__main__":
    root, snr, mid, out = Path(sys.argv[1]), sys.argv[2], sys.argv[3], Path(sys.argv[4])
    d = root / f"{snr}_dB_fan" / "fan" / mid
    files = sorted(d.glob("*/*.wav"))
    with Pool(4) as pool: R = pool.map(one, files, chunksize=8)
    ids = np.array([f.parent.name + "/" + f.name for f in files]); y = np.array([f.parent.name == "abnormal" for f in files], int)
    out.mkdir(parents=True, exist_ok=True)
    np.savez(out / f"lm_{snr}_{mid}.npz", ids=ids, y=y, ch1=np.stack([r[0] for r in R]), ort8=np.stack([r[1] for r in R]), gort8=np.stack([r[2] for r in R]),
             h_ch1=np.array([r[3] for r in R], np.float32), h_ort8=np.array([r[4] for r in R], np.float32), hnames=np.array(R[0][5]))
    print(snr, mid, len(files), "normal", int((y == 0).sum()), "abnormal", int(y.sum()))


In [ ]:
%%writefile /content/code/prep_colab.py
"""Colab için: bir SNR'ın fan zip'ini bütün halde indirir, açmadan içinden okuyarak 4 ID için öznitelik dosyalarını (feat/lm_{snr}_{id}.npz) üretir, zip'i siler.
Kullanım: python prep_colab.py 6 [id_00,id_02,id_04,id_06]"""
import sys, io, os, zipfile, subprocess, numpy as np, soundfile as sf
from pathlib import Path
from multiprocessing import Pool
sys.path.insert(0, str(Path(__file__).parent)); import feats
snr = sys.argv[1]; ids = (sys.argv[2] if len(sys.argv) > 2 else "id_00,id_02,id_04,id_06").split(",")
FE = Path("feat"); FE.mkdir(exist_ok=True)
todo = [m for m in ids if not (FE / f"lm_{snr}_{m}.npz").exists()]
if not todo: print("hazır:", snr); sys.exit()
ZP = Path(f"/content/{snr}_dB_fan.zip")
subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", str(ZP), f"https://zenodo.org/records/3384388/files/{snr}_dB_fan.zip?download=1"], check=True)
def one_member(n):
    with zipfile.ZipFile(ZP) as z: data = z.read(n)
    tmp = f"/dev/shm/{os.getpid()}.wav"; open(tmp, "wb").write(data)
    return feats.one(tmp)
with zipfile.ZipFile(ZP) as z: allnames = sorted(n for n in z.namelist() if n.endswith(".wav"))
for m in todo:
    names = [n for n in allnames if f"/{m}/" in n]
    with Pool(os.cpu_count()) as pool: R = pool.map(one_member, names, chunksize=8)
    ids_ = np.array([n.split("/")[-2] + "/" + n.split("/")[-1] for n in names]); y = np.array([n.split("/")[-2] == "abnormal" for n in names], int)
    np.savez(FE / f"lm_{snr}_{m}.npz", ids=ids_, y=y, ch1=np.stack([r[0] for r in R]), ort8=np.stack([r[1] for r in R]), gort8=np.stack([r[2] for r in R]),
             h_ch1=np.array([r[3] for r in R], np.float32), h_ort8=np.array([r[4] for r in R], np.float32), hnames=np.array(R[0][5]))
    print(snr, m, len(names), "normal", int((y == 0).sum()), "arızalı", int(y.sum()), flush=True)
ZP.unlink()


In [ ]:
%%writefile /content/code/exp.py
import numpy as np, torch, torch.nn as nn, random, time
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
torch.set_num_threads(4)
dev = "cuda" if torch.cuda.is_available() else "cpu"
IDS = ["id_00", "id_02", "id_04", "id_06"]

def auc2(y, s): return roc_auc_score(y, s), roc_auc_score(y, s, max_fpr=0.1)

def split(y, seed):
    """Arızalıların tamamı + aynı sayıda normal test, kalan normaller eğitim. Tohum bölünmeyi değiştirir."""
    rng = np.random.RandomState(seed)
    ni = np.where(y == 0)[0].copy(); rng.shuffle(ni); ai = np.where(y == 1)[0]
    if __import__("os").environ.get("PROTO") == "tez":      # tez protokolü: normallerin %80'i eğitim, %20 normal + tüm arızalı test
        k = int(round(len(ni) * 0.2)); return ni[k:], np.r_[ni[:k], ai]
    return ni[len(ai):], np.r_[ni[:len(ai)], ai]

def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)

# ---------------- Dense AE (Hafta 3-6 temel modeli)
def frames(S, n=5):
    T = S.shape[1] - n + 1
    return np.concatenate([S[:, i:i + T] for i in range(n)], 0).T
class DenseAE(nn.Module):
    def __init__(s, d=640, h=128, z=8):
        super().__init__()
        b = lambda i, o: [nn.Linear(i, o), nn.BatchNorm1d(o), nn.ReLU()]
        s.enc = nn.Sequential(*b(d, h), *b(h, h), *b(h, h), *b(h, h), *b(h, z))
        s.dec = nn.Sequential(*b(z, h), *b(h, h), *b(h, h), *b(h, h), nn.Linear(h, d))
    def forward(s, x): return s.dec(s.enc(x))
def dense_ae(Str, Ste, seed, epochs=30, bs=512):
    seed_all(seed)
    X = np.concatenate([frames(S) for S in Str]).astype(np.float32); mu, sd = X.mean(), X.std()
    X = torch.from_numpy((X - mu) / sd).to(dev); m = DenseAE().to(dev); opt = torch.optim.Adam(m.parameters(), 1e-3)
    for _ in range(epochs):
        m.train(); perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), bs):
            xb = X[perm[i:i + bs]]; loss = ((m(xb) - xb) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return np.array([((m(x) - x) ** 2).mean().item() for x in (torch.from_numpy((frames(S).astype(np.float32) - mu) / sd).to(dev) for S in Ste)])

# ---------------- Bant istatistiği + kNN (etiketsiz, eğitimsiz)
def bandstats(S): S = S.astype(np.float32); return np.r_[S.mean(1), S.std(1)]
def knn_score(Ftr, Fte, k=5):
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
    nn_ = NearestNeighbors(n_neighbors=k).fit((Ftr - mu) / sd)
    return nn_.kneighbors((Fte - mu) / sd)[0].mean(1)

# ---------------- Makine ID sınıflandırıcısı (öz-denetimli)
class IDNet(nn.Module):
    def __init__(s, n_cls, emb=128, w=32):
        super().__init__()
        def blk(i, o, p): return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(p)]
        s.f = nn.Sequential(*blk(1, w, 2), *blk(w, 2 * w, 2), *blk(2 * w, 4 * w, 2), *blk(4 * w, 4 * w, (2, 1)))
        s.emb = nn.Sequential(nn.Linear(4 * w * 8, emb), nn.BatchNorm1d(emb), nn.ReLU()); s.out = nn.Linear(emb, n_cls)
    def embed(s, x):                       # x: B x 128 mel x T
        h = s.f(x.unsqueeze(1))            # B x 128 x 8 x T/8
        return s.emb(h.mean(3).flatten(1))  # zaman üzerinde ortalama, frekans bilgisi korunur
    def forward(s, x): return s.out(s.embed(x))

def specaug(x, F=16, T=12):
    x = x.clone()
    for b in range(x.size(0)):
        f0 = np.random.randint(0, x.size(1) - F); t0 = np.random.randint(0, x.size(2) - T)
        x[b, f0:f0 + np.random.randint(0, F), :] = 0; x[b, :, t0:t0 + np.random.randint(0, T)] = 0
    return x

def id_classifier(train_sets, test_sets, seed, win=64, epochs=40, bs=64, aug=True, w=32):
    """train_sets/test_sets: {sınıf_no: [S,...]}. Döner: {sınıf: (logp_skor, emb_eğitim, emb_test)}"""
    seed_all(seed)
    allS = [S for v in train_sets.values() for S in v]
    mu = np.mean([S.astype(np.float32).mean(1) for S in allS], 0)[:, None]; sd = np.std(np.concatenate([S.astype(np.float32) for S in allS], 1), 1)[:, None] + 1e-6
    norm = lambda S: (S.astype(np.float32) - mu) / sd
    Xtr = [(torch.from_numpy(norm(S)), c) for c, v in train_sets.items() for S in v]
    m = IDNet(len(train_sets), w=w).to(dev); opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=epochs * ((len(Xtr) + bs - 1) // bs))
    ce = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train(); order = np.random.permutation(len(Xtr))
        for i in range(0, len(order), bs):
            idx = order[i:i + bs]; T = Xtr[0][0].shape[1]
            st = np.random.randint(0, T - win, len(idx))
            xb = torch.stack([Xtr[j][0][:, s:s + win] for j, s in zip(idx, st)]).to(dev)
            yb = torch.tensor([Xtr[j][1] for j in idx], device=dev)
            if aug: xb = specaug(xb)
            loss = ce(m(xb), yb); opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    m.eval()
    def clip(S):   # klibi örtüşen pencerelere böl
        x = torch.from_numpy(norm(S)); T = x.shape[1]
        return torch.stack([x[:, s:s + win] for s in range(0, T - win + 1, win // 2)]).to(dev)
    out = {}
    with torch.no_grad():
        for c in test_sets:
            sc, et = [], []
            for S in test_sets[c]:
                xw = clip(S); e = m.embed(xw); lp = torch.log_softmax(m.out(e), 1)[:, c]
                sc.append(-lp.mean().item()); et.append(e.mean(0).cpu().numpy())
            etr = np.array([m.embed(clip(S)).mean(0).cpu().numpy() for S in train_sets[c]])
            out[c] = (np.array(sc), etr, np.array(et))
    return out

# ---------------- ID-XAI: arıza etiketi kullanmadan öznitelik seçimi (önerilen)
def knn_norm(Ftr, Fte, k=5):
    """kNN mesafesi, eğitim normallerinin kendi (kendini dışlayan) kNN mesafesinin medyanına bölünür -> farklı k'lar birleştirilebilir."""
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6; A, B = (Ftr - mu) / sd, (Fte - mu) / sd
    nn_ = NearestNeighbors(n_neighbors=k + 1).fit(A)
    ref = np.median(nn_.kneighbors(A)[0][:, 1:].mean(1))
    return nn_.kneighbors(B, n_neighbors=k)[0].mean(1) / ref
def idxai_scores(H_by_id, tr_by_id, te_by_id, seed, ks=(5, 10, 20, 33)):
    from sklearn.ensemble import RandomForestClassifier
    ids = list(H_by_id); Xn = np.concatenate([H_by_id[m][tr_by_id[m]] for m in ids])
    lab = np.concatenate([[i] * len(tr_by_id[m]) for i, m in enumerate(ids)]); out = {}
    for i, m in enumerate(ids):
        imp = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed, n_jobs=-1).fit(Xn, (lab == i).astype(int)).feature_importances_
        o = np.argsort(-imp); H = H_by_id[m]; r = {}
        for k in ks:
            c = o[:k]; r[f"k{k}"] = knn_norm(H[tr_by_id[m]][:, c], H[te_by_id[m]][:, c])
        r["ens"] = np.mean([r[f"k{k}"] for k in ks], 0); out[m] = (r, o)
    return out


In [ ]:
%%writefile /content/code/run.py
import sys, csv, time, numpy as np
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent)); from exp import *
from sklearn.ensemble import IsolationForest
snr, ids, methods = sys.argv[1], sys.argv[2].split(","), sys.argv[3].split(",")
modes = sys.argv[4].split(",") if len(sys.argv) > 4 else ["ch1", "ort8", "gort8"]
seeds = [int(s) for s in sys.argv[5].split(",")] if len(sys.argv) > 5 else [42, 43, 44]
import os
PROTO = os.environ.get("PROTO", "denge")
FE = Path("feat"); OUT = Path("results.csv" if PROTO == "denge" else "results_tez.csv")
D = {m: np.load(FE / f"lm_{snr}_{m}.npz") for m in ids}
SC = Path("scores" if PROTO == "denge" else "scores_tez"); SC.mkdir(exist_ok=True)
def log(scores=None, y=None, **r):
    if scores is not None:
        r["auc"], r["pauc"] = auc2(y, scores)
        np.savez(SC / f"{r['snr']}_{r['id']}_{r['mode']}_{r['method']}_{r['seed']}.npz", s=scores, y=y)
    new = not OUT.exists()
    with open(OUT, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["snr", "id", "mode", "method", "seed", "auc", "pauc"]); new and w.writeheader(); w.writerow(r)
    print(r, flush=True)
for seed in seeds:
    for mode in modes:
        sp = {m: split(D[m]["y"], seed) for m in ids}
        S = {m: D[m][mode] for m in ids} if mode != "both" else None
        for m in ids:
            tr, te = sp[m]; y = D[m]["y"][te]
            if "knn" in methods:
                F = np.array([bandstats(s) for s in S[m]]); log(snr=snr, id=m, mode=mode, method="bant_knn", seed=seed, scores=knn_score(F[tr], F[te]), y=y)
            if "hand" in methods and mode != "gort8":
                H = D[m]["h_" + mode]; mu, sd = H[tr].mean(0), H[tr].std(0) + 1e-6; Z = (H - mu) / sd
                log(snr=snr, id=m, mode=mode, method="el_knn", seed=seed, scores=knn_score(H[tr], H[te]), y=y)
                ifs = -IsolationForest(n_estimators=200, random_state=seed).fit(Z[tr]).score_samples(Z[te])
                log(snr=snr, id=m, mode=mode, method="el_if", seed=seed, scores=ifs, y=y)
            if "dense" in methods:
                sc = dense_ae([S[m][i] for i in tr], [S[m][i] for i in te], seed)
                log(snr=snr, id=m, mode=mode, method="dense_ae", seed=seed, scores=sc, y=y)
        if "idxai" in methods and mode in ("ch1", "ort8", "both"):
            getH = lambda m: np.concatenate([D[m]["h_ch1"], D[m]["h_ort8"]], 1) if mode == "both" else D[m]["h_" + mode]
            out = idxai_scores({m: getH(m) for m in ids}, {m: sp[m][0] for m in ids}, {m: sp[m][1] for m in ids}, seed)
            for m in ids:
                y = D[m]["y"][sp[m][1]]
                for kk, sc in out[m][0].items(): log(snr=snr, id=m, mode=mode, method=f"idxai_{kk}", seed=seed, scores=sc, y=y)
        if "idc" in methods and len(ids) > 1:
            t0 = time.time()
            out = id_classifier({c: [S[m][i] for i in sp[m][0]] for c, m in enumerate(ids)},
                                {c: [S[m][i] for i in sp[m][1]] for c, m in enumerate(ids)}, seed, epochs=int(__import__("os").environ.get("IDC_EP", 40)), w=int(__import__("os").environ.get("IDC_W", 32)))
            for c, m in enumerate(ids):
                y = D[m]["y"][sp[m][1]]; sc, etr, ete = out[c]
                log(snr=snr, id=m, mode=mode, method="id_sinif", seed=seed, scores=sc, y=y)
                log(snr=snr, id=m, mode=mode, method="id_emb_knn", seed=seed, scores=knn_score(etr, ete), y=y)
            print("idc süre", int(time.time() - t0), flush=True)


In [ ]:
%%writefile /content/code/sel.py
"""XAI ile öznitelik seçimi + gözetimsiz detektör, 4 fan ID x 3 SNR.
Seçiciler:
  etiket_F  : arıza etiketli seçim (tezdeki gibi) ama ayrı seçim kümesinde; test hiç görülmez (yarı gözetimli, dürüst)
  etiket_tez: tez protokolü; seçim tüm etiketli veriyle, en iyi k test AUC'sine göre (iyimser, tezle birebir kıyas)
  id_xai    : YENİ, arıza etiketi kullanmaz. RF yalnızca normal kayıtlarla 'bu ses bu ID mi?' (bire karşı diğerleri) öğrenir,
              önem sıralamasından ilk k öznitelik seçilir.
Detektörler: kNN, Isolation Forest, autoencoder (Hafta 7 FeatAE)."""
import sys, csv, numpy as np, torch, torch.nn as nn
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score
torch.set_num_threads(2)
IDS = ["id_00", "id_02", "id_04", "id_06"]; KS = [5, 10, 20, 33, 60]
snrs = sys.argv[1].split(","); seeds = [int(s) for s in sys.argv[2].split(",")]; FEAT = sys.argv[3] if len(sys.argv) > 3 else "h_ch1"
import os
DROP = os.environ.get("DROP_LEVEL") == "1"; KS = [int(k) for k in os.environ.get("KS", ",".join(map(str, KS))).split(",")]
PROTOS = os.environ.get("PROTOS", "F,tez").split(",")
OUT = Path(f"sel_{FEAT}{'_seviyesiz' if DROP else ''}.csv")
SEVIYE = ("rms", "db", "envelope", "hilbert", "magnitude", "psd", "cqt")
def auc2(y, s): return round(roc_auc_score(y, s), 4), round(roc_auc_score(y, s, max_fpr=0.1), 4)
class FeatAE(nn.Module):
    def __init__(s, d, h=64, z=8):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.1), nn.Linear(h, z), nn.ReLU()); s.dec = nn.Sequential(nn.Linear(z, h), nn.ReLU(), nn.Linear(h, d))
    def forward(s, x): return s.dec(s.enc(x))
def det_ae(Xtr, Xev, seed, epochs=100):
    torch.manual_seed(seed); sc = MinMaxScaler().fit(Xtr); A = torch.tensor(sc.transform(Xtr), dtype=torch.float32)
    m = FeatAE(A.shape[1]); opt = torch.optim.Adam(m.parameters(), 1e-3); g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        m.train(); perm = torch.randperm(len(A), generator=g)
        for i in range(0, len(A), 32):
            xb = A[perm[i:i + 32]]; loss = ((m(xb) - xb) ** 2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad(): return [((m(B) - B) ** 2).mean(1).numpy() for B in (torch.tensor(sc.transform(X), dtype=torch.float32) for X in Xev)]
def det_if(Xtr, Xev, seed):
    m = IsolationForest(n_estimators=200, random_state=seed).fit(Xtr); return [-m.score_samples(X) for X in Xev]
def det_knn(Xtr, Xev, seed, k=5):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6; nn_ = NearestNeighbors(n_neighbors=k).fit((Xtr - mu) / sd)
    return [nn_.kneighbors((X - mu) / sd)[0].mean(1) for X in Xev]
DETS = {"knn": det_knn, "if": det_if, "ae": det_ae}
def rf_imp(X, y, seed):
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=seed, n_jobs=2).fit(X, y).feature_importances_
def prot_F(y, rng):
    ni, ai = np.where(y == 0)[0], np.where(y == 1)[0]; ni = rng.permutation(ni); ai = rng.permutation(ai); h = len(ai) // 2
    return dict(train=ni[2 * h + (len(ai) - 2 * h):], sel=np.r_[ni[h:len(ai)], ai[h:]], test=np.r_[ni[:h], ai[:h]])
def prot_tez(y, rng):
    ni = rng.permutation(np.where(y == 0)[0]); k = int(round(len(ni) * 0.2)); return dict(train=ni[k:], test=np.r_[ni[:k], np.where(y == 1)[0]])
def log(**r):
    new = not OUT.exists()
    with open(OUT, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["snr", "id", "seed", "protocol", "selector", "k", "detector", "sel_auc", "auc", "pauc"]); new and w.writeheader(); w.writerow(r)
for snr in snrs:
    D = {}
    for m in IDS:
        z = np.load(f"feat/lm_{snr}_{m}.npz"); X = z[FEAT]
        if DROP: X = X[:, [i for i, n in enumerate(z["hnames"]) if not str(n).startswith(SEVIYE)]]
        D[m] = {"y": z["y"], FEAT: X}
    for seed in seeds:
        for proto, pf in [(n, f) for n, f in (("F", prot_F), ("tez", prot_tez)) if n in PROTOS]:
            SP = {m: pf(D[m]["y"], np.random.RandomState(seed)) for m in IDS}
            # ID-XAI: tüm ID'lerin eğitim normalleri, bire karşı diğerleri
            Xn = np.concatenate([D[m][FEAT][SP[m]["train"]] for m in IDS]); idl = np.concatenate([[i] * len(SP[m]["train"]) for i, m in enumerate(IDS)])
            for i, m in enumerate(IDS):
                X, y, sp = D[m][FEAT], D[m]["y"], SP[m]
                imps = {"tum": None, "id_xai": rf_imp(Xn, (idl == i).astype(int), seed)}
                if proto == "F": imps["etiket_F"] = rf_imp(X[np.r_[sp["train"], sp["sel"]]], y[np.r_[sp["train"], sp["sel"]]], seed)
                else: imps["etiket_tez"] = rf_imp(X, y, seed)          # tezdeki gibi tüm etiketli veri (test dahil)
                for selname, imp in imps.items():
                    for k in ([X.shape[1]] if imp is None else KS):
                        cols = np.arange(X.shape[1]) if imp is None else np.argsort(-imp)[:k]
                        ev = [X[sp["test"]][:, cols]] + ([X[sp["sel"]][:, cols]] if "sel" in sp else [])
                        for dn, fn in DETS.items():
                            out = fn(X[sp["train"]][:, cols], ev, seed)
                            sa = auc2(y[sp["sel"]], out[1])[0] if "sel" in sp else ""
                            a, p = auc2(y[sp["test"]], out[0])
                            log(snr=snr, id=m, seed=seed, protocol=proto, selector=selname, k=k, detector=dn, sel_auc=sa, auc=a, pauc=p)
            print(snr, seed, proto, "tamam", flush=True)


In [ ]:
%%writefile /content/code/analyze.py
"""Sonuç tabloları + skor füzyonu. Kullanım: python analyze.py [denge|tez]"""
import sys, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
proto = sys.argv[1] if len(sys.argv) > 1 else "denge"
SC = Path("scores" if proto == "denge" else "scores_tez")
rows = []
for f in SC.glob("*.npz"):
    snr, mid1, mid2, mode, *meth, seed = f.stem.split("_")
    d = np.load(f); rows.append(dict(snr=snr, id=mid1 + "_" + mid2, mode=mode, method="_".join(meth), seed=int(seed), s=d["s"], y=d["y"]))
D = pd.DataFrame(rows)
def get(snr, mid, mode, meth, seed):
    r = D[(D.snr == snr) & (D.id == mid) & (D["mode"] == mode) & (D.method == meth) & (D.seed == seed)]
    return (r.iloc[0].s, r.iloc[0].y) if len(r) else (None, None)
# füzyon: etiket kullanmadan, test skorlarının sıra (rank) ortalaması
FUS = {"fuzyon(id_sinif+el_knn)": [("ch1", "id_sinif"), ("ch1", "el_knn")],
       "fuzyon(id_sinif_ch1+id_sinif_gort8+el_knn)": [("ch1", "id_sinif"), ("gort8", "id_sinif"), ("ch1", "el_knn")],
       "fuzyon(id_sinif+idxai_ens_both)": [("ch1", "id_sinif"), ("both", "idxai_ens")]}
for (snr, mid, seed), _ in D.groupby(["snr", "id", "seed"]):
    for name, parts in FUS.items():
        ss = [get(snr, mid, mo, me, seed) for mo, me in parts]
        if any(s is None for s, _ in ss): continue
        s = np.mean([rankdata(s) / len(s) for s, _ in ss], 0)
        rows.append(dict(snr=snr, id=mid, mode="-", method=name, seed=seed, s=s, y=ss[0][1]))
D = pd.DataFrame(rows)
D["auc"] = [roc_auc_score(y, s) for y, s in zip(D.y, D.s)]
D["pauc"] = [roc_auc_score(y, s, max_fpr=0.1) for y, s in zip(D.y, D.s)]
D.drop(columns=["s", "y"]).to_csv(f"ozet_{proto}.csv", index=False)
for met in ("auc", "pauc"):
    g = D.groupby(["method", "mode", "snr", "id"])[met].mean().unstack("id")
    g["ort"] = g.mean(axis=1)
    print(f"\n==== {proto} | {met} (tohum ortalaması) ====")
    print(g.round(3).to_string())


In [ ]:
%%writefile /content/code/final_tables.py
"""Tüm sonuç tabloları (markdown + csv) ve istatistik testleri."""
import numpy as np, pandas as pd
from scipy.stats import wilcoxon
from pathlib import Path
O = Path("tablolar"); O.mkdir(exist_ok=True)
SN = ["6", "0", "-6"]; md = []
def fmt(g):  # g: DataFrame index rows, cols SN, values (mean,std)
    return g
def summ(d, met="auc"):
    """önce 4 ID ortalaması (tohum başına), sonra tohumlar üzerinden ortalama ± std"""
    a = d.groupby(["snr", "seed"])[met].mean().groupby("snr").agg(["mean", "std"]).reindex(SN)
    return [f"{m:.3f} ± {s:.3f}" if not np.isnan(s) else f"{m:.3f}" for m, s in zip(a["mean"], a["std"])]
TABLES = {}
def table(title, rows, cols=("Yöntem", "+6 dB", "0 dB", "−6 dB")):
    TABLES[title.split(":")[0]] = dict(title=title, cols=list(cols), rows=[list(map(str, r)) for r in rows])
    md.append(f"\n### {title}\n\n| " + " | ".join(cols) + " |\n|" + "---|" * len(cols))
    for r in rows: md.append("| " + " | ".join(map(str, r)) + " |")
D = pd.read_csv("ozet_denge.csv", dtype={"snr": str})
T = pd.read_csv("ozet_tez.csv", dtype={"snr": str}) if Path("ozet_tez.csv").exists() else None
S = pd.read_csv("sel_h_ch1.csv", dtype={"snr": str})
def sel_best(S, sel, det, proto="F", by="sel_auc"):
    x = S[(S.protocol == proto) & (S.selector == sel) & (S.detector == det)]
    return x.loc[x.groupby(["snr", "id", "seed"])[by].idxmax()]
MAIN = [("Dense AE, log-mel (Hafta 3-6 temel)", "dense_ae", "ch1"),
        ("El yapımı 145 öznitelik + kNN", "el_knn", "ch1"),
        ("Log-mel bant istatistiği + kNN, 8 kanal güç ort.", "bant_knn", "gort8"),
        ("Makine ID sınıflandırıcısı (CNN)", "id_sinif", "ch1"),
        ("ID-XAI + kNN topluluğu, 1. kanal", "idxai_ens", "ch1"),
        ("ID-XAI + kNN topluluğu, 1. kanal + 8 kanal ort. (önerilen)", "idxai_ens", "both"),
        ("Füzyon: ID sınıflandırıcısı + önerilen", "fuzyon(id_sinif+idxai_ens_both)", "-")]
for met in ("auc", "pauc"):
    rows = [(n, *summ(D[(D.method == m) & (D["mode"] == mo)], met)) for n, m, mo in MAIN if len(D[(D.method == m) & (D["mode"] == mo)])]
    for det in ("knn", "ae"):
        b = sel_best(S, "etiket_F", det); rows.append((f"Referans (yarı gözetimli): arıza etiketli XAI seçimi + {det.upper()}, Protokol F", *summ(b, met)))
    table(f"Tablo A ({met.upper()}): dengeli protokol, 4 fan ID ortalaması, 3 tohum (ort. ± std)", rows)
# ID bazında önerilen
x = D[(D.method == "idxai_ens") & (D["mode"] == "both")]
t = x.groupby(["id", "snr"]).auc.mean().unstack()[SN]
table("Tablo B: önerilen yöntem, ID bazında AUC (3 tohum ort.)", [(i, *[f"{v:.3f}" for v in r]) for i, r in t.iterrows()] + [("Ortalama", *[f"{v:.3f}" for v in t.mean()])], ("ID", "+6 dB", "0 dB", "−6 dB"))
# literatür
LIT = [("MIMII baseline AE [Purohit 2019]", .913, .795, .663), ("AE [MemNMF Tablo III]", .953, .849, .687), ("VAE", .948, .849, .715), ("GRLNet", .953, .866, .699),
       ("MemNMF-768 (2026)", .929, .873, .764), ("Li vd. 2024, LNS-AE", .950, .862, .716), ("Li vd. 2024, en iyi (TM-LGS)", .971, .902, .762)]
ours = x.groupby(["snr", "seed"]).auc.mean().groupby("snr").mean().reindex(SN)
table("Tablo C: literatürle kıyas (fan, 4 ID ortalaması, etiketsiz, AUC)", [(n, a, b, c) for n, a, b, c in LIT] + [("Bu çalışma: ID-XAI topluluğu (etiketsiz)", *[f"{v:.3f}" for v in ours])])
# tez kıyası (+6 dB, ID bazında)
tz = {"id_00": .9880, "id_02": .9998, "id_04": .9936, "id_06": 1.0}
rows = [("Tez: XAI seçimli AE (etiketli seçim)", *[f"{tz[i]:.4f}" for i in tz], f"{np.mean(list(tz.values())):.4f}")]
for det in ("ae", "knn", "if"):
    b = sel_best(S, "etiket_tez", det, "tez", "auc"); b = b[b.snr == "6"].groupby("id").auc.mean()
    rows.append((f"Bu çalışma, tez protokolü, etiketli seçim + {det.upper()} (en iyi k)", *[f"{b[i]:.4f}" for i in tz], f"{b.mean():.4f}"))
if T is not None:
    for mo, lab in (("both", "1. kanal + 8 kanal ort."), ("ort8", "8 kanal ort."), ("ch1", "1. kanal")):
        y = T[(T.method == "idxai_ens") & (T["mode"] == mo) & (T.snr == "6")].groupby("id").auc.mean()
        if len(y): rows.append((f"Bu çalışma, tez protokolü, ETİKETSİZ ID-XAI ({lab})", *[f"{y[i]:.4f}" for i in tz], f"{y.mean():.4f}"))
table("Tablo D: +6 dB'de tezle kıyas (tez protokolü, AUC)", rows, ("Yöntem", "id_00", "id_02", "id_04", "id_06", "Ort."))
# ablasyonlar: k ve detektör (Protokol F, id_xai)
x = S[(S.protocol == "F") & (S.selector.isin(["id_xai", "tum"]))]
g = x.groupby(["selector", "k", "detector", "snr", "seed"]).auc.mean().groupby(["selector", "k", "detector", "snr"]).mean().unstack("snr")[SN]
table("Tablo E: ID-XAI ablasyonu (Protokol F, 4 ID ort., 3 tohum): öznitelik sayısı k ve detektör", [(f"{'tüm öznitelik' if s == 'tum' else 'ID-XAI'}, k={k}, {d.upper()}", *[f"{v:.3f}" for v in r]) for (s, k, d), r in g.iterrows()])
for tn, f, lab in (("F1", "sel_h_ch1_seviyesiz.csv", "seviye öznitelikleri çıkarılmış (89 öznitelik)"), ("F2", "sel_h_ort8.csv", "8 kanal ortalaması öznitelikleri")):
    if not Path(f).exists(): continue
    z = pd.read_csv(f, dtype={"snr": str}); z = z[z.protocol == "F"]
    g = z.groupby(["selector", "k", "detector", "snr", "seed"]).auc.mean().groupby(["selector", "k", "detector", "snr"]).mean().unstack("snr").reindex(columns=SN)
    table(f"Tablo {tn}: {lab}, Protokol F", [(f"{s}, k={k}, {d.upper()}", *[f"{v:.3f}" for v in r]) for (s, k, d), r in g.iterrows()])
# Wilcoxon: önerilen vs diğerleri (ID x tohum çiftleri)
rows = []
P = D[(D.method == "idxai_ens") & (D["mode"] == "both")].set_index(["snr", "id", "seed"]).auc
for n, m, mo in MAIN:
    if m == "idxai_ens" and mo == "both": continue
    Q = D[(D.method == m) & (D["mode"] == mo)].set_index(["snr", "id", "seed"]).auc
    r = [n]
    for s in SN:
        a, b = P.loc[s].align(Q.loc[s] if s in Q.index.get_level_values(0) else Q.iloc[:0], join="inner") if s in Q.index.get_level_values(0) else (None, None)
        if a is None or len(a) < 5: r.append("-"); continue
        diff = (a - b).mean(); p = wilcoxon(a, b).pvalue if (a != b).any() else 1.0
        r.append(f"{diff:+.3f} (p={p:.3g}, n={len(a)})")
    rows.append(r)
table("Tablo G: önerilen yönteme göre AUC farkı (önerilen − yöntem), Wilcoxon işaretli sıra testi, ID×tohum çiftleri", rows)
Path("tablolar/tablolar.md").write_text("\n".join(md)); print("\n".join(md))
import json; Path("tablolar/tablolar.json").write_text(json.dumps(TABLES, ensure_ascii=False, indent=1))


In [ ]:
%%writefile /content/code/plots.py
"""Hafta 9-10 şekilleri."""
import pandas as pd, numpy as np, matplotlib, matplotlib.ticker; matplotlib.use("Agg"); import matplotlib.pyplot as plt
d = pd.read_csv("ozet_denge.csv", dtype={"snr": str}); SN = ["6", "0", "-6"]; X = [6, 0, -6]
SEL = [("dense_ae", "ch1", "Dense AE, log-mel (önceki temel)", "#9aa0a6", "o"),
       ("el_knn", "ch1", "145 öznitelik + kNN (seçimsiz)", "#e8710a", "s"),
       ("id_sinif", "ch1", "Makine ID sınıflandırıcısı (CNN)", "#1a73e8", "^"),
       ("idxai_ens", "both", "ID-XAI + kNN topluluğu (önerilen)", "#188038", "D")]
LIT = {"MIMII baseline AE": [.913, .795, .663], "MemNMF (2026)": [.929, .873, .764], "Li vd. (2024) en iyi": [.971, .902, .762]}
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.5))
for k, met in enumerate(["auc", "pauc"]):
    for meth, mode, lab, c, mk in SEL:
        r = d[(d.method == meth) & (d["mode"] == mode)]
        if r.empty: continue
        g = r.groupby(["snr", "seed"])[met].mean().groupby("snr").agg(["mean", "std"]).reindex(SN)
        ax[k].errorbar(X, g["mean"], yerr=g["std"].fillna(0), label=lab, color=c, marker=mk, capsize=3, lw=2)
    if met == "auc":
        for (n, v), ls in zip(LIT.items(), [":", "-.", "--"]): ax[k].plot(X, v, ls, color="k", lw=1, alpha=.6, label=f"Literatür: {n}")
    ax[k].axhline(0.5, ls="--", c="grey", lw=0.7); ax[k].set_xticks(X); ax[k].set_xticklabels(["+6 dB", "0 dB", "−6 dB"]); ax[k].invert_xaxis()
    ax[k].set_ylim(0.45, 1.01); ax[k].set_title(("AUC" if met == "auc" else "pAUC (FPR ≤ 0,1)") + ", 4 fan ID ortalaması"); ax[k].grid(alpha=.3); ax[k].set_xlabel("SNR (sağa doğru gürültü artıyor)")
ax[0].legend(fontsize=7.5, loc="lower left"); plt.tight_layout(); plt.savefig("sekil8_hafta9.png", dpi=170)
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
for k, (meth, mode, lab, _, _) in enumerate(SEL):
    r = d[(d.method == meth) & (d["mode"] == mode)]
    t = r.groupby(["id", "snr"]).auc.mean().unstack().reindex(columns=SN)
    im = ax[k].imshow(t.values, vmin=0.5, vmax=1, cmap="viridis")
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.values[i, j]
            if not np.isnan(v): ax[k].text(j, i, f"{v:.2f}", ha="center", va="center", color="w" if v < .8 else "k", fontsize=9)
    ax[k].set_xticks(range(3)); ax[k].set_xticklabels(["+6", "0", "−6"]); ax[k].set_yticks(range(len(t))); ax[k].set_yticklabels(t.index); ax[k].set_title(lab, fontsize=8.5)
fig.colorbar(im, ax=ax, shrink=.85, label="AUC"); plt.savefig("sekil9_hafta9.png", dpi=170, bbox_inches="tight")
S = pd.read_csv("sel_h_ch1.csv", dtype={"snr": str}); S = S[(S.protocol == "F") & (S.detector == "knn")]
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for j, s in enumerate(SN):
    for sel, lab, c in (("etiket_F", "Arıza etiketli XAI seçimi (yarı gözetimli)", "#d93025"), ("id_xai", "ID-XAI (etiketsiz, önerilen)", "#188038")):
        g = S[(S.snr == s) & (S.selector == sel)].groupby(["k", "seed"]).auc.mean().groupby("k").agg(["mean", "std"])
        ax[j].errorbar(g.index, g["mean"], yerr=g["std"], marker="o", capsize=3, color=c, label=lab)
    a = S[(S.snr == s) & (S.selector == "tum")].groupby("seed").auc.mean().mean()
    ax[j].axhline(a, color="#e8710a", ls="--", label="Tüm 145 öznitelik"); ax[j].set_xscale("log"); ax[j].set_xticks([5, 10, 20, 33, 60]); ax[j].set_xticklabels([5, 10, 20, 33, 60]); ax[j].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    ax[j].set_title(f"{'+' if s == '6' else ''}{s} dB"); ax[j].set_xlabel("seçilen öznitelik sayısı k"); ax[j].grid(alpha=.3)
ax[0].set_ylabel("AUC (kNN, Protokol F)"); ax[0].legend(fontsize=7.5); plt.tight_layout(); plt.savefig("sekil10_hafta10.png", dpi=170)
print("şekiller kaydedildi")


## 1. Veri hazırlığı (SNR başına ~10 GB indirme, sonra silinir; Drive'a yalnızca öznitelikler yazılır, SNR başına ~1,3 GB)

In [ ]:
for snr in ['6', '0', '-6']:
    !python /content/code/prep_colab.py {snr}
!ls -la feat

## 2. Deneyler
GPU'da ID sınıflandırıcısı tam genişlikte (w=32, 40 epoch) çalışır. Sonuç satırları `results.csv`, klip skorları `scores/` altına yazılır.

In [ ]:
A = 'id_00,id_02,id_04,id_06'
for snr in ['6', '0', '-6']:
    !python /content/code/run.py {snr} {A} knn,hand ch1,ort8,gort8 42,43,44
    !python /content/code/run.py {snr} {A} idxai ch1,ort8,both 42,43,44
    !IDC_W=32 IDC_EP=40 python /content/code/run.py {snr} {A} idc ch1,gort8 42,43,44
    !python /content/code/run.py {snr} {A} dense ch1 42

In [ ]:
# Tez protokolü (normallerin %80'i eğitim; %20 normal + tüm arızalılar test) — tez Tablo 4.6 / 4.16 ile kıyas için
for snr in ['6', '0', '-6']:
    !PROTO=tez python /content/code/run.py {snr} {A} knn,hand,idxai ch1,ort8,both 42,43,44
    !PROTO=tez IDC_W=32 IDC_EP=40 python /content/code/run.py {snr} {A} idc ch1 42,43,44

## 3. XAI öznitelik seçimi: arıza etiketli (tez yöntemi) ve etiketsiz ID-XAI, k ve detektör taraması + ablasyonlar

In [ ]:
!python /content/code/sel.py 6,0,-6 42,43,44 h_ch1
!DROP_LEVEL=1 PROTOS=F KS=5,10,20 python /content/code/sel.py 6,0,-6 42,43,44 h_ch1
!PROTOS=F KS=5,10,20 python /content/code/sel.py 6,0,-6 42,43,44 h_ort8

## 4. Tablolar, istatistik testleri ve şekiller

In [ ]:
!python /content/code/analyze.py denge > /dev/null
!python /content/code/analyze.py tez > /dev/null
!python /content/code/final_tables.py
!python /content/code/plots.py
from IPython.display import Image, display
display(Image('sekil8_hafta9.png')); display(Image('sekil9_hafta9.png')); display(Image('sekil10_hafta10.png'))